# 1. Data Loading

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/Video_Games_Sales_as_at_22_Dec_2016.csv")

print(df.head())
print(df.info())
print(df.isnull().sum())

                       Name Platform  Year_of_Release         Genre Publisher  \
0                Wii Sports      Wii           2006.0        Sports  Nintendo   
1         Super Mario Bros.      NES           1985.0      Platform  Nintendo   
2            Mario Kart Wii      Wii           2008.0        Racing  Nintendo   
3         Wii Sports Resort      Wii           2009.0        Sports  Nintendo   
4  Pokemon Red/Pokemon Blue       GB           1996.0  Role-Playing  Nintendo   

   NA_Sales  EU_Sales  JP_Sales  Other_Sales  Global_Sales  Critic_Score  \
0     41.36     28.96      3.77         8.45         82.53          76.0   
1     29.08      3.58      6.81         0.77         40.24           NaN   
2     15.68     12.76      3.79         3.29         35.52          82.0   
3     15.61     10.93      3.28         2.95         32.77          80.0   
4     11.27      8.89     10.22         1.00         31.37           NaN   

   Critic_Count  User_Score  User_Count Developer Rating

# 2. Data Cleaning and Preprocessing

In [2]:
clean_df = df.copy()

clean_df = clean_df.dropna(subset=['Name', 'Genre', 'Year_of_Release'])

clean_df['Publisher'] = clean_df['Publisher'].fillna('Unknown')
clean_df['Developer'] = clean_df['Developer'].fillna('Unknown')
clean_df['Rating'] = clean_df['Rating'].fillna('Unknown')

clean_df['User_Score'] = pd.to_numeric(clean_df['User_Score'], errors='coerce')

print(clean_df.isnull().sum())
print("Original rows:", df.shape)
print("Clean rows:", clean_df.shape)

zero_sales = clean_df[
    (clean_df['NA_Sales'] == 0) &
    (clean_df['EU_Sales'] == 0) &
    (clean_df['JP_Sales'] == 0) &
    (clean_df['Other_Sales'] == 0) &
    (clean_df['Global_Sales'] == 0)
]

print("Rows with zero sales:", zero_sales.shape)

Name                  0
Platform              0
Year_of_Release       0
Genre                 0
Publisher             0
NA_Sales              0
EU_Sales              0
JP_Sales              0
Other_Sales           0
Global_Sales          0
Critic_Score       8465
Critic_Count       8465
User_Score         8985
User_Count         8985
Developer             0
Rating                0
dtype: int64
Original rows: (16719, 16)
Clean rows: (16448, 16)
Rows with zero sales: (0, 16)


In [3]:
clean_df = clean_df[
    ~(
        (clean_df['NA_Sales'] == 0) &
        (clean_df['EU_Sales'] == 0) &
        (clean_df['JP_Sales'] == 0) &
        (clean_df['Other_Sales'] == 0) &
        (clean_df['Global_Sales'] == 0)
    )
]

In [4]:
df_2000 = clean_df[
    (clean_df['Year_of_Release'] >= 2000) &
    (clean_df['Year_of_Release'] <= 2016)
].copy()

In [5]:
df_2000 = df_2000.sort_values('Global_Sales', ascending=False)

df_2000 = df_2000.drop_duplicates(
    subset=['Name', 'Platform', 'Year_of_Release'],
    keep='first'
)

print("Dataset after filtering and duplicate removal:", df_2000.shape)

Dataset after filtering and duplicate removal: (14469, 16)


In [9]:
df_genre_df_2000 = df_2000.copy()

df_genre_df_2000['Critic_Score'] = df_genre_df_2000.groupby('Genre')['Critic_Score']\
    .transform(lambda x: x.fillna(x.median()))

df_genre_df_2000['User_Score'] = df_genre_df_2000.groupby('Genre')['User_Score']\
    .transform(lambda x: x.fillna(x.median()))

df_genre_df_2000['Critic_Count'] = df_genre_df_2000.groupby('Genre')['Critic_Count']\
    .transform(lambda x: x.fillna(x.median()))

df_genre_df_2000['User_Count'] = df_genre_df_2000.groupby('Genre')['User_Count']\
    .transform(lambda x: x.fillna(x.median()))

print(df_genre_df_2000.isnull().sum())
print("Final cleaned dataset:", df_genre_df_2000.shape)

Name               0
Platform           0
Year_of_Release    0
Genre              0
Publisher          0
NA_Sales           0
EU_Sales           0
JP_Sales           0
Other_Sales        0
Global_Sales       0
Critic_Score       0
Critic_Count       0
User_Score         0
User_Count         0
Developer          0
Rating             0
dtype: int64
Final cleaned dataset: (14469, 16)


In [10]:
df_genre_df_2000[['EU_Sales', 'Critic_Score', 'User_Score']].corr()

,EU_Sales,Critic_Score,User_Score
EU_Sales,1.000000,0.190986,0.021507
Critic_Score,0.190986,1.000000,0.465876
User_Score,0.021507,0.465876,1.000000


In [11]:
df_2000[['Global_Sales', 'Critic_Score', 'User_Score']].corr()

,Global_Sales,Critic_Score,User_Score
Global_Sales,1.00000,0.238200,0.080940
Critic_Score,0.23820,1.000000,0.576482
User_Score,0.08094,0.576482,1.000000


# 3. Statistical Analysis

In [12]:
from scipy.stats import f_oneway

genres = df_2000.groupby('Genre')['Global_Sales'].apply(list)
f_oneway(*genres)

F_onewayResult(statistic=np.float64(15.76579653105253), pvalue=np.float64(3.7015671676742587e-31))

In [13]:
rating = df_2000.groupby('Rating')['Global_Sales'].apply(list)
f_oneway(*rating)

F_onewayResult(statistic=np.float64(39.53930364622835), pvalue=np.float64(1.9304669508509343e-55))

In [14]:
platform = df_2000.groupby('Platform')['Global_Sales'].apply(list)
f_oneway(*platform)

F_onewayResult(statistic=np.float64(13.946767209995967), pvalue=np.float64(8.048195314544975e-45))

In [15]:
from scipy.stats import chi2_contingency

table = pd.crosstab(df_2000['Genre'], df_2000['Rating'])
chi2_contingency(table)

Chi2ContingencyResult(statistic=np.float64(6756.118729496955), pvalue=np.float64(0.0), dof=77, expected_freq=array([[2.12799779e-01, 8.25663142e+02, 2.96217292e+02, 1.70239823e+00,
        3.23668464e+02, 2.12799779e-01, 6.08394568e+02, 1.02292854e+03],
       [8.25212523e-02, 3.20182459e+02, 1.14869583e+02, 6.60170019e-01,
        1.25514825e+02, 8.25212523e-02, 2.35928260e+02, 3.96679660e+02],
       [4.42324971e-02, 1.71622089e+02, 6.15716359e+01, 3.53859977e-01,
        6.72776280e+01, 4.42324971e-02, 1.26460709e+02, 2.12625613e+02],
       [1.10373903e-01, 4.28250743e+02, 1.53640473e+02, 8.82991223e-01,
        1.67878706e+02, 1.10373903e-01, 3.15558988e+02, 5.30567351e+02],
       [4.97615592e-02, 1.93074850e+02, 6.92680904e+01, 3.98092474e-01,
        7.56873315e+01, 4.97615592e-02, 1.42268298e+02, 2.39203815e+02],
       [3.31052595e-02, 1.28448407e+02, 4.60825213e+01, 2.64842076e-01,
        5.03530997e+01, 3.31052595e-02, 9.46479370e+01, 1.59136983e+02],
       [7.15322413e-0

# 4. Regression Analysis

In [16]:
import statsmodels.api as sm

X = df_genre_df_2000[['Critic_Score', 'User_Score']]
y = df_genre_df_2000['Global_Sales']

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:           Global_Sales   R-squared:                       0.048
Model:                            OLS   Adj. R-squared:                  0.048
Method:                 Least Squares   F-statistic:                     366.4
Date:                Wed, 03 Jun 2026   Prob (F-statistic):          6.00e-156
Time:                        11:09:35   Log-Likelihood:                -25683.
No. Observations:               14469   AIC:                         5.137e+04
Df Residuals:                   14466   BIC:                         5.139e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const           -1.2460      0.093    -13.390   

In [17]:
df_genre_df_2000['Log_Global_Sales'] = np.log(df_genre_df_2000['Global_Sales'])

In [18]:
X = df_genre_df_2000[['Critic_Score', 'User_Score']]
y = df_genre_df_2000['Log_Global_Sales']

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:       Log_Global_Sales   R-squared:                       0.077
Model:                            OLS   Adj. R-squared:                  0.077
Method:                 Least Squares   F-statistic:                     607.6
Date:                Wed, 03 Jun 2026   Prob (F-statistic):          4.11e-254
Time:                        11:10:07   Log-Likelihood:                -25406.
No. Observations:               14469   AIC:                         5.082e+04
Df Residuals:                   14466   BIC:                         5.084e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const           -3.7953      0.091    -41.572   

In [19]:
import statsmodels.formula.api as smf

model = smf.ols(
    'Log_Global_Sales ~ Critic_Score + User_Score + C(Genre) + C(Platform) + C(Rating)',
    data=df_genre_df_2000
).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:       Log_Global_Sales   R-squared:                       0.284
Model:                            OLS   Adj. R-squared:                  0.282
Method:                 Least Squares   F-statistic:                     146.7
Date:                Wed, 03 Jun 2026   Prob (F-statistic):               0.00
Time:                        11:10:15   Log-Likelihood:                -23573.
No. Observations:               14469   AIC:                         4.723e+04
Df Residuals:                   14429   BIC:                         4.753e+04
Df Model:                          39                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

# 5. Machine Learning Models

In [20]:
def group_platform(platform):
    if platform in ['PS', 'PS2', 'PS3', 'PS4', 'PSP', 'PSV']:
        return 'PlayStation'
    elif platform in ['XB', 'X360', 'XOne']:
        return 'Xbox'
    elif platform in ['Wii', 'WiiU', 'DS', '3DS', 'GB', 'GBA', 'GC', 'N64']:
        return 'Nintendo'
    elif platform == 'PC':
        return 'PC'
    else:
        return 'Other'

df_genre_df_2000['Platform_Group'] = df_genre_df_2000['Platform'].apply(group_platform)

df_model1 = df_genre_df_2000.copy()

df_model1['Log_Global_Sales'] = np.log1p(df_model1['Global_Sales'])

print(df_model1.head())
print("Dataset shape:", df_model1.shape)

                    Name Platform  Year_of_Release     Genre Publisher  \
0             Wii Sports      Wii           2006.0    Sports  Nintendo   
2         Mario Kart Wii      Wii           2008.0    Racing  Nintendo   
3      Wii Sports Resort      Wii           2009.0    Sports  Nintendo   
6  New Super Mario Bros.       DS           2006.0  Platform  Nintendo   
7               Wii Play      Wii           2006.0      Misc  Nintendo   

   NA_Sales  EU_Sales  JP_Sales  Other_Sales  Global_Sales  Critic_Score  \
0     41.36     28.96      3.77         8.45         82.53          76.0   
2     15.68     12.76      3.79         3.29         35.52          82.0   
3     15.61     10.93      3.28         2.95         32.77          80.0   
6     11.28      9.14      6.50         2.88         29.80          89.0   
7     13.96      9.18      2.93         2.84         28.92          58.0   

   Critic_Count  User_Score  User_Count Developer Rating  Log_Global_Sales  \
0          51.0     

In [21]:
df_ml = pd.get_dummies(
    df_model1,
    columns=['Genre', 'Platform_Group', 'Rating'],
    drop_first=True
)

print(df_ml.head())
print("Encoded dataset shape:", df_ml.shape)

                    Name Platform  Year_of_Release Publisher  NA_Sales  \
0             Wii Sports      Wii           2006.0  Nintendo     41.36   
2         Mario Kart Wii      Wii           2008.0  Nintendo     15.68   
3      Wii Sports Resort      Wii           2009.0  Nintendo     15.61   
6  New Super Mario Bros.       DS           2006.0  Nintendo     11.28   
7               Wii Play      Wii           2006.0  Nintendo     13.96   

   EU_Sales  JP_Sales  Other_Sales  Global_Sales  Critic_Score  ...  \
0     28.96      3.77         8.45         82.53          76.0  ...   
2     12.76      3.79         3.29         35.52          82.0  ...   
3     10.93      3.28         2.95         32.77          80.0  ...   
6      9.14      6.50         2.88         29.80          89.0  ...   
7      9.18      2.93         2.84         28.92          58.0  ...   

   Platform_Group_PC  Platform_Group_PlayStation  Platform_Group_Xbox  \
0              False                       False       

In [22]:
X = df_ml.drop([
    'Name',
    'Publisher',
    'Developer',
    'Global_Sales',
    'Log_Global_Sales',
    'NA_Sales',
    'EU_Sales',
    'JP_Sales',
    'Other_Sales',
    'Critic_Count',
    'User_Count',
    'Platform'
], axis=1)

y = df_ml['Log_Global_Sales']

print("Number of features:", len(X.columns))
print(X.columns[:20])

Number of features: 25
Index(['Year_of_Release', 'Critic_Score', 'User_Score', 'Genre_Adventure',
       'Genre_Fighting', 'Genre_Misc', 'Genre_Platform', 'Genre_Puzzle',
       'Genre_Racing', 'Genre_Role-Playing', 'Genre_Shooter',
       'Genre_Simulation', 'Genre_Sports', 'Genre_Strategy',
       'Platform_Group_Other', 'Platform_Group_PC',
       'Platform_Group_PlayStation', 'Platform_Group_Xbox', 'Rating_E',
       'Rating_E10+'],
      dtype='object')


In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

Training set shape: (11575, 25)
Testing set shape: (2894, 25)


In [24]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)

rf_model.fit(X_train, y_train)

print("Random Forest model trained successfully.")

y_train_pred = rf_model.predict(X_train)

train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
train_mae = mean_absolute_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)

print("\nTraining Results:")
print("RMSE:", round(train_rmse, 4))
print("MAE:", round(train_mae, 4))
print("R²:", round(train_r2, 4))

Random Forest model trained successfully.

Training Results:
RMSE: 0.2822
MAE: 0.1817
R²: 0.4463


In [25]:
y_pred = rf_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Model Evaluation Results:")
print("RMSE:", round(rmse, 4))
print("MAE:", round(mae, 4))
print("R²:", round(r2, 4))

Model Evaluation Results:
RMSE: 0.3059
MAE: 0.1955
R²: 0.379


In [26]:
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

nn_model = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPRegressor(
        hidden_layer_sizes=(32, 16, 16),
        activation='relu',
        solver='adam',
        alpha=0.0001,
        learning_rate_init=0.0005,
        max_iter=2000,
        random_state=42
    ))
])

nn_model.fit(X_train, y_train)

y_train_pred = nn_model.predict(X_train)
y_test_pred = nn_model.predict(X_test)

print("Train RMSE:", np.sqrt(mean_squared_error(y_train, y_train_pred)))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, y_test_pred)))
print("Train MAE:", mean_absolute_error(y_train, y_train_pred))
print("Test MAE:", mean_absolute_error(y_test, y_test_pred))
print("Train R2:", r2_score(y_train, y_train_pred))
print("Test R2:", r2_score(y_test, y_test_pred))

Train RMSE: 0.2947015288276257
Test RMSE: 0.30743487236615047
Train MAE: 0.18956859547379418
Test MAE: 0.19925014508791253
Train R2: 0.3960442010712443
Test R2: 0.37291391285024755


# 6. Final CatBoost Model

In [27]:
from catboost import CatBoostRegressor

df_cat = df_genre_df_2000.copy()

df_cat['Log_Global_Sales'] = np.log1p(df_cat['Global_Sales'])

X_cat = df_cat.drop([
    'Name',
    'Global_Sales',
    'Log_Global_Sales',
    'NA_Sales',
    'EU_Sales',
    'JP_Sales',
    'Other_Sales',
    'Platform'
], axis=1).copy()

y_cat = df_cat['Log_Global_Sales'].copy()

categorical_cols_cat = [col for col in X_cat.columns if X_cat[col].dtype == 'object']

for col in X_cat.columns:
    if col in categorical_cols_cat:
        X_cat[col] = X_cat[col].fillna('Unknown').astype(str)
    else:
        X_cat[col] = X_cat[col].fillna(X_cat[col].median())

if 'Year_of_Release' in X_cat.columns:
    X_cat['Year_of_Release'] = X_cat['Year_of_Release'].astype(int)

print("Feature data types:")
print(X_cat.dtypes)

print("\nCategorical columns used by CatBoost:")
print(categorical_cols_cat)

X_train_cat, X_test_cat, y_train_cat, y_test_cat = train_test_split(
    X_cat,
    y_cat,
    test_size=0.2,
    random_state=42
)

print("\nTraining set shape:", X_train_cat.shape)
print("Testing set shape:", X_test_cat.shape)

cat_model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=5,
    loss_function='RMSE',
    eval_metric='RMSE',
    random_seed=42,
    verbose=0
)

cat_model.fit(
    X_train_cat,
    y_train_cat,
    cat_features=categorical_cols_cat,
    eval_set=(X_test_cat, y_test_cat),
    use_best_model=True
)

print("\nCatBoost model trained successfully.")

y_train_pred_cat = cat_model.predict(X_train_cat)
y_test_pred_cat = cat_model.predict(X_test_cat)

train_rmse_cat = np.sqrt(mean_squared_error(y_train_cat, y_train_pred_cat))
train_mae_cat = mean_absolute_error(y_train_cat, y_train_pred_cat)
train_r2_cat = r2_score(y_train_cat, y_train_pred_cat)

print("\nTraining Results:")
print("RMSE:", round(train_rmse_cat, 4))
print("MAE:", round(train_mae_cat, 4))
print("R²:", round(train_r2_cat, 4))

test_rmse_cat = np.sqrt(mean_squared_error(y_test_cat, y_test_pred_cat))
test_mae_cat = mean_absolute_error(y_test_cat, y_test_pred_cat)
test_r2_cat = r2_score(y_test_cat, y_test_pred_cat)

print("\nModel Evaluation Results:")
print("RMSE:", round(test_rmse_cat, 4))
print("MAE:", round(test_mae_cat, 4))
print("R²:", round(test_r2_cat, 4))

Feature data types:
Year_of_Release      int64
Genre               object
Publisher           object
Critic_Score       float64
Critic_Count       float64
User_Score         float64
User_Count         float64
Developer           object
Rating              object
Platform_Group      object
dtype: object

Categorical columns used by CatBoost:
['Genre', 'Publisher', 'Developer', 'Rating', 'Platform_Group']

Training set shape: (11575, 10)
Testing set shape: (2894, 10)

CatBoost model trained successfully.

Training Results:
RMSE: 0.2155
MAE: 0.1339
R²: 0.6771

Model Evaluation Results:
RMSE: 0.2448
MAE: 0.1506
R²: 0.6025


In [29]:
cat_importance = pd.Series(
    cat_model.get_feature_importance(),
    index=X_train_cat.columns
).sort_values(ascending=False)

print("Top 15 Important Features:")
print(cat_importance.head(15))


Top 15 Important Features:
User_Count         24.304320
Publisher          20.292135
Platform_Group     15.627705
Year_of_Release     9.627337
Critic_Score        7.575264
Genre               5.671655
Rating              5.023119
Critic_Count        4.632724
Developer           4.474987
User_Score          2.770755
dtype: float64


# 7. Recommendation System

In [48]:
def build_market_intelligence(base_df, focal_publisher):
    df = base_df.copy()
    total_titles = len(df)

    genre_demand = df.groupby('Genre')['Global_Sales'].mean()
    genre_titles = df.groupby('Genre').size()
    competition_share = (genre_titles / total_titles) * 100

    focal_games = df[df['Publisher'] == focal_publisher]
    focal_genre_titles = focal_games.groupby('Genre').size()
    focal_market_share = ((focal_genre_titles / genre_titles) * 100).fillna(0)

    competitor_sales = (
        df[df['Publisher'] != focal_publisher]
        .groupby(['Genre', 'Publisher'])['Global_Sales']
        .sum()
        .reset_index()
    )

    top_competitor_by_genre = (
        competitor_sales
        .sort_values(['Genre', 'Global_Sales'], ascending=[True, False])
        .groupby('Genre')
        .first()
        .reset_index()
        .rename(columns={
            'Publisher': 'Top_Competitor',
            'Global_Sales': 'Top_Competitor_Sales'
        })
    )

    market_info = pd.DataFrame({
        'Genre': genre_demand.index,
        'Market_Demand': genre_demand.values,
        'Titles_in_Genre': genre_titles.values,
        'Competition_Share_%': competition_share.values,
        'Focal_Publisher_Share_%': [
            focal_market_share.get(g, 0) for g in genre_demand.index
        ]
    })

    q1 = market_info['Competition_Share_%'].quantile(0.33)
    q2 = market_info['Competition_Share_%'].quantile(0.66)

    def classify_competition_level(x):
        if x <= q1:
            return 'Low'
        elif x <= q2:
            return 'Moderate'
        else:
            return 'High'

    market_info['Competition_Level'] = market_info['Competition_Share_%'].apply(classify_competition_level)

    demand_q2 = market_info['Market_Demand'].quantile(0.66)

    def strategic_position(row):
        high_demand = row['Market_Demand'] >= demand_q2
        low_comp = row['Competition_Level'] == 'Low'
        moderate_comp = row['Competition_Level'] == 'Moderate'

        if high_demand and low_comp:
            return 'High Opportunity'
        elif high_demand and moderate_comp:
            return 'Competitive Opportunity'
        elif not high_demand and row['Competition_Level'] == 'High':
            return 'Saturated / Low Priority'
        else:
            return 'Selective Opportunity'

    market_info['Strategic_Position'] = market_info.apply(strategic_position, axis=1)

    market_info = market_info.merge(
        top_competitor_by_genre,
        on='Genre',
        how='left'
    )

    market_info['Top_Competitor'] = market_info['Top_Competitor'].fillna('No major competitor identified')
    market_info['Top_Competitor_Sales'] = market_info['Top_Competitor_Sales'].fillna(0)

    return market_info

In [49]:
def build_regional_demand(base_df):
    regional = base_df.groupby('Genre')[['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales']].mean().reset_index()
    regional['Best_Region'] = regional[['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales']].idxmax(axis=1)
    return regional


def build_sales_thresholds(base_df):
    q1 = base_df['Global_Sales'].quantile(0.33)
    q2 = base_df['Global_Sales'].quantile(0.66)
    q3 = base_df['Global_Sales'].quantile(0.80)

    return {
        'low_to_medium': q1,
        'medium_to_high': q2,
        'very_high': q3
    }

In [50]:
def predict_game_concept_catboost(
    genre,
    platform_group,
    rating,
    year_of_release,
    critic_score,
    user_score,
    model,
    feature_columns,
    base_df,
    publisher='Unknown',
    developer='Unknown'
):
    new_game = {}

    for col in feature_columns:
        if col == 'Genre':
            new_game[col] = str(genre)

        elif col == 'Platform_Group':
            new_game[col] = str(platform_group)

        elif col == 'Rating':
            new_game[col] = str(rating)

        elif col == 'Year_of_Release':
            new_game[col] = int(year_of_release)

        elif col == 'Critic_Score':
            new_game[col] = float(critic_score)

        elif col == 'User_Score':
            new_game[col] = float(user_score)

        elif col == 'Critic_Count':
            new_game[col] = float(base_df['Critic_Count'].median())

        elif col == 'User_Count':
            new_game[col] = float(base_df['User_Count'].median())

        elif col == 'Publisher':
            new_game[col] = str(publisher)

        elif col == 'Developer':
            new_game[col] = str(developer)

        else:
            if col in base_df.columns:
                if base_df[col].dtype == 'object':
                    new_game[col] = str(base_df[col].mode()[0])
                else:
                    new_game[col] = float(base_df[col].median())
            else:
                new_game[col] = 0

    new_game_df = pd.DataFrame([new_game])

    for col in new_game_df.select_dtypes(include='object').columns:
        new_game_df[col] = new_game_df[col].astype(str)

    predicted_log_sales = model.predict(new_game_df)[0]
    predicted_sales = np.expm1(predicted_log_sales)

    return predicted_log_sales, predicted_sales

In [52]:
def classify_sales_potential(predicted_sales, thresholds):
    if predicted_sales >= thresholds['medium_to_high']:
        return 'High'
    elif predicted_sales >= thresholds['low_to_medium']:
        return 'Medium'
    else:
        return 'Low'


def classify_risk_level(predicted_sales, sales_potential, competition_level, focal_share):
    if sales_potential == 'Low' and competition_level == 'High':
        return 'High Risk'
    elif sales_potential == 'Medium' and competition_level == 'High':
        return 'Moderate Risk'
    elif focal_share >= 20 and sales_potential != 'Low':
        return 'Low Risk'
    else:
        return 'Moderate Risk'


def calculate_opportunity_score(
    predicted_sales,
    critic_score,
    user_score,
    market_demand,
    competition_share,
    focal_share,
    sales_thresholds,
    market_table
):
    max_demand = market_table['Market_Demand'].max()
    max_comp_share = market_table['Competition_Share_%'].max()
    high_sales_ref = sales_thresholds['very_high'] if sales_thresholds['very_high'] > 0 else 1

    sales_score = min(predicted_sales / high_sales_ref, 1) * 35
    demand_score = min(market_demand / max_demand, 1) * 25
    critic_score_part = (critic_score / 100) * 15
    user_score_part = (user_score / 10) * 10
    competition_penalty = min(competition_share / max_comp_share, 1) * 10
    saturation_penalty = min(focal_share / 100, 1) * 5

    opportunity_score = (
        sales_score +
        demand_score +
        critic_score_part +
        user_score_part -
        competition_penalty -
        saturation_penalty
    )

    opportunity_score = max(0, min(100, opportunity_score))
    return round(opportunity_score, 2)


def classify_opportunity_band(score):
    if score >= 70:
        return 'High Opportunity'
    elif score >= 45:
        return 'Moderate Opportunity'
    else:
        return 'Low Opportunity'


def region_strategy(best_region):
    if best_region == 'NA_Sales':
        return 'Prioritize North America as the primary target market.'
    elif best_region == 'EU_Sales':
        return 'Prioritize Europe as the primary target market.'
    elif best_region == 'JP_Sales':
        return 'Prioritize Japan as the primary target market.'
    elif best_region == 'Other_Sales':
        return 'Prioritize secondary international markets.'
    else:
        return 'No clear regional priority identified.'

In [58]:
def recommend_strategy_catboost_final(
    genre,
    platform_group,
    rating,
    year_of_release,
    critic_score,
    user_score,
    model,
    feature_columns,
    market_table,
    regional_table,
    base_df,
    sales_thresholds,
    publisher='Unknown',
    developer='Unknown'
):
    if genre not in market_table['Genre'].values:
        raise ValueError(f"Genre '{genre}' not found in market_table")

    if genre not in regional_table['Genre'].values:
        raise ValueError(f"Genre '{genre}' not found in regional_table")

    predicted_log_sales, predicted_sales = predict_game_concept_catboost(
        genre=genre,
        platform_group=platform_group,
        rating=rating,
        year_of_release=year_of_release,
        critic_score=critic_score,
        user_score=user_score,
        model=model,
        feature_columns=feature_columns,
        base_df=base_df,
        publisher=publisher,
        developer=developer
    )

    genre_market = market_table[market_table['Genre'] == genre].iloc[0]
    genre_region = regional_table[regional_table['Genre'] == genre].iloc[0]

    sales_potential = classify_sales_potential(predicted_sales, sales_thresholds)

    risk = classify_risk_level(
        predicted_sales=predicted_sales,
        sales_potential=sales_potential,
        competition_level=genre_market['Competition_Level'],
        focal_share=genre_market['Focal_Publisher_Share_%']
    )

    opportunity_score = calculate_opportunity_score(
        predicted_sales=predicted_sales,
        critic_score=critic_score,
        user_score=user_score,
        market_demand=genre_market['Market_Demand'],
        competition_share=genre_market['Competition_Share_%'],
        focal_share=genre_market['Focal_Publisher_Share_%'],
        sales_thresholds=sales_thresholds,
        market_table=market_table
    )

    opportunity_band = classify_opportunity_band(opportunity_score)

    best_region = genre_region['Best_Region']
    region_recommendation = region_strategy(best_region)

    if opportunity_score >= 70 and risk != 'High Risk':
        final_action = 'Recommended for market entry'
    elif opportunity_score >= 45:
        final_action = 'Recommended with caution'
    else:
        final_action = 'Not recommended as a priority'

    recommendation = {
        'Genre': genre,
        'Platform_Group': platform_group,
        'Rating': rating,
        'Year_of_Release': year_of_release,
        'Publisher_Input': publisher,
        'Developer_Input': developer,
        'Expected_Critic_Score': critic_score,
        'Expected_User_Score': user_score,
        'Predicted_Log_Sales': round(predicted_log_sales, 4),
        'Predicted_Global_Sales': round(predicted_sales, 4),
        'Sales_Potential': sales_potential,

        'Market_Demand': round(genre_market['Market_Demand'], 4),
        'Competition_Share_%': round(genre_market['Competition_Share_%'], 2),
        'Competition_Level': genre_market['Competition_Level'],
        'Focal_Publisher_Share_%': round(genre_market['Focal_Publisher_Share_%'], 2),
        'Strategic_Position': genre_market['Strategic_Position'],

        'Top_Competitor': genre_market['Top_Competitor'],
        'Top_Competitor_Sales': round(genre_market['Top_Competitor_Sales'], 2),

        'Best_Region': best_region,
        'Region_Strategy': region_recommendation,

        'Risk_Level': risk,
        'Opportunity_Score': opportunity_score,
        'Opportunity_Band': opportunity_band,
        'Final_Action': final_action
    }

    return recommendation

# 8. Scenario Testing

In [59]:
focal_publisher = "Electronic Arts"

market_info_final = build_market_intelligence(
    base_df=df_genre_df_2000,
    focal_publisher=focal_publisher
)

regional_demand_final = build_regional_demand(df_genre_df_2000)

sales_thresholds_final = build_sales_thresholds(df_genre_df_2000)

In [60]:
example_recommendation = recommend_strategy_catboost_final(
    genre='Sports',
    platform_group='PlayStation',
    rating='E',
    year_of_release=2017,
    critic_score=88,
    user_score=8.4,
    model=cat_model,
    feature_columns=X_train_cat.columns,
    market_table=market_info_final,
    regional_table=regional_demand_final,
    base_df=df_genre_df_2000,
    sales_thresholds=sales_thresholds_final,
    publisher='Electronic Arts',
    developer='EA Canada'
)

example_recommendation

{'Genre': 'Sports',
 'Platform_Group': 'PlayStation',
 'Rating': 'E',
 'Year_of_Release': 2017,
 'Publisher_Input': 'Electronic Arts',
 'Developer_Input': 'EA Canada',
 'Expected_Critic_Score': 88,
 'Expected_User_Score': 8.4,
 'Predicted_Log_Sales': np.float64(0.4984),
 'Predicted_Global_Sales': np.float64(0.6461),
 'Sales_Potential': 'High',
 'Market_Demand': np.float64(0.5721),
 'Competition_Share_%': np.float64(13.67),
 'Competition_Level': 'High',
 'Focal_Publisher_Share_%': np.float64(25.83),
 'Strategic_Position': 'Selective Opportunity',
 'Top_Competitor': 'Nintendo',
 'Top_Competitor_Sales': np.float64(185.91),
 'Best_Region': 'NA_Sales',
 'Region_Strategy': 'Prioritize North America as the primary target market.',
 'Risk_Level': 'Low Risk',
 'Opportunity_Score': np.float64(66.57),
 'Opportunity_Band': 'Moderate Opportunity',
 'Final_Action': 'Recommended with caution'}

In [61]:
recommendation_df = pd.DataFrame([example_recommendation]).T
recommendation_df.columns = ["Value"]

recommendation_df

,Value
Genre,Sports
Platform_Group,PlayStation
Rating,E
Year_of_Release,2017
Publisher_Input,Electronic Arts
Developer_Input,EA Canada
Expected_Critic_Score,88
Expected_User_Score,8.4
Predicted_Log_Sales,0.4984
Predicted_Global_Sales,0.6461


In [62]:
example_recommendation = recommend_strategy_catboost_final(
    genre='Adventure',
    platform_group='PC',
    rating='E',
    year_of_release=2017,
    critic_score=62,
    user_score=6.8,
    model=cat_model,
    feature_columns=X_train_cat.columns,
    market_table=market_info_final,
    regional_table=regional_demand_final,
    base_df=df_genre_df_2000,
    sales_thresholds=sales_thresholds_final,
    publisher='Electronic Arts',
    developer='EA Canada'
)

example_recommendation

{'Genre': 'Adventure',
 'Platform_Group': 'PC',
 'Rating': 'E',
 'Year_of_Release': 2017,
 'Publisher_Input': 'Electronic Arts',
 'Developer_Input': 'EA Canada',
 'Expected_Critic_Score': 62,
 'Expected_User_Score': 6.8,
 'Predicted_Log_Sales': np.float64(-0.018),
 'Predicted_Global_Sales': np.float64(-0.0179),
 'Sales_Potential': 'Low',
 'Market_Demand': np.float64(0.154),
 'Competition_Share_%': np.float64(8.25),
 'Competition_Level': 'Moderate',
 'Focal_Publisher_Share_%': np.float64(0.84),
 'Strategic_Position': 'Selective Opportunity',
 'Top_Competitor': 'Ubisoft',
 'Top_Competitor_Sales': np.float64(22.12),
 'Best_Region': 'NA_Sales',
 'Region_Strategy': 'Prioritize North America as the primary target market.',
 'Risk_Level': 'Moderate Risk',
 'Opportunity_Score': np.float64(15.82),
 'Opportunity_Band': 'Low Opportunity',
 'Final_Action': 'Not recommended as a priority'}

In [63]:
recommendation_df = pd.DataFrame([example_recommendation]).T
recommendation_df.columns = ["Value"]

recommendation_df

,Value
Genre,Adventure
Platform_Group,PC
Rating,E
Year_of_Release,2017
Publisher_Input,Electronic Arts
Developer_Input,EA Canada
Expected_Critic_Score,62
Expected_User_Score,6.8
Predicted_Log_Sales,-0.018
Predicted_Global_Sales,-0.0179
